In [76]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns 
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import duckdb
import pandas as pd

In [77]:
con = duckdb.connect("health_enforcement.duckdb")
df_mart = con.execute("SELECT * FROM enriched_mart").df()

In [78]:
severity_map = {
    'AA'        : 6.0,  # direct proximate cause of death — $25k-$100k fine
    'A TREBLED' : 4.5,
    'A'         : 4.0,  # imminent danger of death or serious harm
    'B TREBLED' : 3.0,
    'B FIRST'   : 2.5,
    'B'         : 2.0,  # direct relationship to health/safety
    'AP IJ'     : 4.0,  # hospital immediate jeopardy — up to $75k-$125k
    'AP NON-IJ' : 2.5,  # hospital non-IJ — up to $25k
    'AP BR'     : 2.0,
    'AP NHPPD'  : 1.5,  # staffing standard violation
    'FTR AE'    : 2.0,
    'FRTR RES'  : 1.5,
    'FTR BR'    : 1.5,
    'FTR RES'   : 1.5,
    'WF'        : 2.5,  # falsification — integrity violation, bump up
    'WO'        : 2.5,  # omission — same
    'NP'        : 0.5,
    'RD'        : 1.0,
}

# Apply to CLASS_ASSESSED_FINAL
df_mart['FINAL_SEVERITY'] = df_mart['CLASS_ASSESSED_FINAL'].map(severity_map)

severe_classes = ['AA', 'AP NON-IJ']
df_mart['severe_event'] = df_mart['CLASS_ASSESSED_FINAL'].isin(severe_classes).astype(int)
# UPHELD inherits initial class severity
upheld_mask = df_mart['CLASS_ASSESSED_FINAL'] == 'UPHELD'
df_mart.loc[upheld_mask, 'FINAL_SEVERITY'] = (
    df_mart.loc[upheld_mask, 'CLASS_ASSESSED_INITIAL'].map(severity_map)
)

In [79]:
citations_per_visit = df_mart.groupby(['FACID','EVENTID']).agg(
    facility_name           = ('FACILITY_NAME', 'first'),
    total_action_per_visit  = ('FACILITY_NAME', 'count'),
    location                = ('DISTRICT_OFFICE', 'first'),
    severity_per_action     = ('FINAL_SEVERITY', 'sum'),
    total_deaths            = ('DEATH_RELATED', 'sum'),
    is_low_resource         = ('IS_LOW_RESOURCE', 'first'),
    ltc                     = ('LTC', 'first'),
    appealed                = ('APPEALED', 'sum'),
    balance_due             = ('TOTAL_BALANCE_DUE', 'first'),
    total_offset            = ('TOTAL_PENALTY_OFFSET_AMOUNT', 'first'),
    complaints              = ('COMPLAINT_COUNT', 'sum'),
    last_penalty_date       = ('PENALTY_ISSUE_DATE', 'max'),
).reset_index()

data_end = df_mart['PENALTY_ISSUE_DATE'].max()
citations_per_visit['days_since_last_penalty'] = (
    pd.Timestamp.today() - citations_per_visit['last_penalty_date']
).dt.days

citations_per_visit.head(2)

,FACID,EVENTID,facility_name,total_action_per_visit,location,severity_per_action,total_deaths,is_low_resource,ltc,appealed,balance_due,total_offset,complaints,last_penalty_date,days_since_last_penalty
0,10000001,1VMB11,VINEYARD POST ACUTE,1,Santa Rosa,2.0,0,0,LTC,0,0.0,-1050.0,1,2008-12-23,6291
1,10000001,61KU11,VINEYARD POST ACUTE,1,Santa Rosa,4.0,0,0,LTC,0,0.0,-3150.0,1,2008-09-22,6383


In [80]:
facility_summary = citations_per_visit.groupby('FACID').agg(
    facility_name           = ('facility_name', 'first'),
    total_visits            = ('EVENTID', 'count'),
    avg_citations_per_visit = ('total_action_per_visit', 'mean'),
    total_citations         = ('total_action_per_visit', 'sum'),
    total_deaths            = ('total_deaths', 'sum'),
    is_low_resource         = ('is_low_resource', 'first'),
    ltc                     = ('ltc', 'first'),
    total_appealed          = ('appealed', 'sum'),
    total_balance_due       = ('balance_due', 'sum'),
    total_offset            = ('total_offset', 'sum'),
    total_complaints        = ('complaints', 'sum'),
    avg_severity_per_visit  = ('severity_per_action', 'mean'),
    total_severity          = ('severity_per_action', 'sum'),
    days_since_last_penalty = ('days_since_last_penalty', 'min'),  # most recent across all visits
).reset_index()

facility_summary['appeal_rate']       = facility_summary['total_appealed']   / facility_summary['total_citations']
facility_summary['death_rate']        = facility_summary['total_deaths']      / facility_summary['total_citations']
facility_summary['complaint_rate']    = facility_summary['total_complaints']  / facility_summary['total_visits']
facility_summary['balance_per_visit'] = facility_summary['total_balance_due']
facility_summary['risk_recency'] = facility_summary['days_since_last_penalty'].rank(pct=True)

facility_summary.head(2)

,FACID,facility_name,total_visits,avg_citations_per_visit,total_citations,total_deaths,is_low_resource,ltc,total_appealed,total_balance_due,total_offset,total_complaints,avg_severity_per_visit,total_severity,days_since_last_penalty,appeal_rate,death_rate,complaint_rate,balance_per_visit,risk_recency
0,10000001,VINEYARD POST ACUTE,19,1.368421,26,1,0,LTC,8,0.0,-14000.0,27,3.526316,67.0,818,0.307692,0.038462,1.421053,0.0,0.164207
1,10000003,CREEKSIDE REHABILITATION & BEHAVIORAL HEALTH,17,2.117647,36,2,0,LTC,18,25000.0,-8350.0,16,4.029412,68.5,640,0.500000,0.055556,0.941176,25000.0,0.015498


In [81]:
facility_summary['risk_deaths'] = facility_summary['total_deaths'].rank(pct=True)
facility_summary['risk_severity'] = facility_summary['avg_severity_per_visit'].rank(pct=True)
facility_summary['risk_citations'] = facility_summary['avg_citations_per_visit'].rank(pct=True)
facility_summary['risk_complaints'] = facility_summary['total_complaints'].rank(pct=True)
facility_summary['risk_balance'] = facility_summary['total_balance_due'].rank(pct=True)
facility_summary['risk_recency'] = facility_summary['total_balance_due'].rank(pct=True)
citations_per_visit['bad_visit'] = (
    citations_per_visit['severity_per_action'] >
    citations_per_visit['severity_per_action'].quantile(0.75)
).astype(int)

In [84]:
facility_summary['risk_score'] = (
    facility_summary['risk_deaths'] +
    facility_summary['risk_severity'] +
    facility_summary['risk_balance'] +
    facility_summary['risk_complaints']
)

In [85]:
top_25 = (
    facility_summary
    .sort_values('risk_score', ascending=False)
    .head(25)[[
        'FACID',
        'facility_name',
        'risk_score',
        'death_rate',
        'avg_severity_per_visit',
        'days_since_last_penalty'
    ]]
)

print(top_25)

          FACID                                      facility_name  \
2648  970000054                      WESTERN CONVALESCENT HOSPITAL   
2079  910000326    WEST HOLLYWOOD HEALTHCARE & WELLNESS CENTRE, LP   
2290  940000041                              VILLA DEL RIO GARDENS   
322    40000061                            GRACE HEALTHCARE CENTER   
2657  970000077                         SOUTH PASADENA CARE CENTER   
1336  170001857     DEPARTMENT OF STATE HOSPITALS - PATTON D/P ICF   
1105  110000046                            NAPA VALLEY CARE CENTER   
2056  910000069                               PLAYA DEL REY CENTER   
3      10000005                                RIDGEWAY POST ACUTE   
1743  250000148                                OAK GLEN POST ACUTE   
1466  230000029  CHICO HEIGHTS REHABILITATION & WELLNESS CENTRE...   
702    70000036                          WHITE BLOSSOM CARE CENTER   
205    30000046                      WINDSOR EL CAMINO CARE CENTER   
1      10000003     

past history

In [86]:
df_mart.columns

Index(['EVENTID', 'FACID', 'FACILITY_NAME', 'FAC_TYPE_CODE', 'FAC_FDR',
       'DISTRICT_OFFICE', 'LTC', 'FACILITY_TYPE_DESC', 'IS_HOSPITAL',
       'IS_24HR', 'IS_LOW_RESOURCE', 'PENALTY_NUMBER', 'PENALTY_ISSUE_DATE',
       'PENALTY_TYPE', 'PENALTY_CATEGORY', 'DISPOSITION',
       'CLASS_ASSESSED_INITIAL', 'CLASS_ASSESSED_FINAL',
       'CLASS_ASSESSED_FINAL_DESC', 'DEATH_RELATED', 'APPEALED',
       'TOTAL_AMOUNT_INITIAL', 'TOTAL_AMOUNT_DUE_FINAL', 'TOTAL_BALANCE_DUE',
       'TOTAL_PENALTY_OFFSET_AMOUNT', 'HIGHEST_PRIORITY', 'COMPLAINT_COUNT',
       'FINAL_SEVERITY', 'severe_event'],
      dtype='str')

In [87]:
first_severe = (
    df_mart[df_mart['severe_event'] == 1]
    .sort_values('PENALTY_ISSUE_DATE')
    .groupby('FACID')
    .first()
    .reset_index()[['FACID','PENALTY_ISSUE_DATE']]
)

first_severe.rename(columns={'PENALTY_ISSUE_DATE':'first_severe_date'}, inplace=True)

In [88]:
first_severe

,FACID,first_severe_date
0,10000005,2007-06-25
1,10000024,2000-08-02
2,10000026,2004-07-09
3,10000028,2003-11-14
4,10000042,2003-11-06
...,...,...
332,960002310,2019-05-06
333,970000052,2021-10-18
334,970000111,2023-01-20
335,970000117,2008-01-31


In [89]:
df_mart = df_mart.merge(first_severe, on='FACID', how='left')

In [90]:
prior_history = df_mart[
    (df_mart['PENALTY_ISSUE_DATE'] < df_mart['first_severe_date'])
]
prior_citations = (
    prior_history.groupby('FACID')
    .size()
    .reset_index(name='prior_citations')
)
prior_severity = (
    prior_history.groupby('FACID')['FINAL_SEVERITY']
    .sum()
    .reset_index(name='prior_severity')
)
prior_complaints = (
    prior_history.groupby('FACID')['COMPLAINT_COUNT']
    .sum()
    .reset_index(name='prior_complaints')
)

In [91]:
facility_summary['had_severe_event'] = (
    facility_summary['FACID'].isin(first_severe['FACID'])
).astype(int)
facility_summary.groupby('had_severe_event')[[
    'avg_citations_per_visit',
    'avg_severity_per_visit',
    'total_complaints'
]].mean()

,avg_citations_per_visit,avg_severity_per_visit,total_complaints
had_severe_event,,,
0,1.483586,3.134259,5.879899
1,1.499226,4.131292,14.590504


In [92]:
'PENALTY_ISSUE_DATE'

'PENALTY_ISSUE_DATE'

In [93]:
time_to_severe = (
    df_mart[df_mart['severe_event'] == 1]
    .groupby('FACID')['PENALTY_ISSUE_DATE']
    .min()
)
time_to_severe

FACID
10000005    2007-06-25
10000024    2000-08-02
10000026    2004-07-09
10000028    2003-11-14
10000042    2003-11-06
               ...    
960002310   2019-05-06
970000052   2021-10-18
970000111   2023-01-20
970000117   2008-01-31
970000156   2022-10-06
Name: PENALTY_ISSUE_DATE, Length: 337, dtype: datetime64[us]

In [94]:
first_event = df_mart.groupby('FACID')['PENALTY_ISSUE_DATE'].min()

lead_time = (
    first_severe.set_index('FACID')['first_severe_date']
    - first_event
).dt.days
lead_time.describe()

count     337.000000
mean     2756.059347
std      2251.191072
min         0.000000
25%       620.000000
50%      2656.000000
75%      3906.000000
max      8611.000000
dtype: float64

In [47]:
df_mart['severe_event'].sum()

np.int64(505)

In [52]:
first_complaint = (
    df_mart[df_mart['COMPLAINT_COUNT'] > 0]
    .groupby('FACID')['PENALTY_ISSUE_DATE']
    .min()
)

complaint_lead_clean = complaint_lead[complaint_lead >= 0]

complaint_lead_clean.describe()

count     272.000000
mean     2255.003676
std      1800.410346
min         0.000000
25%       213.750000
50%      2464.500000
75%      3622.000000
max      5996.000000
dtype: float64